# Timeseries processing over a period

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os
import shutil
from pprint import pprint

In [ ]:
from sweat.api import read_ts_input_data, run_timeseries, run_window_time_series
from sweat.timeseries.timeseries_handler import create_config, window_generator

In [ ]:
import logging

from sweat.logging import LoggerManager

LoggerManager.set_level(logging.INFO)

In [ ]:
def locate_test_data() -> str:
    """
    Locate test data directory
    """
    current = os.getcwd()
    parent = os.path.dirname(current)
    if os.path.isdir(os.path.join(parent, "tests")):
        return os.path.join(parent, "tests", "data", "timeseries")
    parent = os.path.dirname(parent)
    if os.path.isdir(os.path.join(parent, "tests")):
        return os.path.join(parent, "tests", "data", "timeseries")
    msg = "Test data directory not found"
    raise FileNotFoundError(msg)


def clean(ts, config):
    """
    Clean outputs
    """
    # Remove directories
    shutil.rmtree(config, ignore_errors=True)
    shutil.rmtree(ts, ignore_errors=True)
    # Recreate directories
    os.makedirs(config, exist_ok=True)
    os.makedirs(ts, exist_ok=True)

### Input

In [ ]:
min_date_str = "2025-08-23"
max_date_str = "2025-08-29"

In [ ]:
min_date_dt = dt.datetime.strptime(min_date_str, "%Y-%m-%d")  # noqa: DTZ007
max_date_dt = dt.datetime.strptime(max_date_str, "%Y-%m-%d")  # noqa: DTZ007

Input directories

In [ ]:
et_single_date_dir = locate_test_data()
radiation_dir = locate_test_data()

Output directories

In [ ]:
output_path = "out"
os.makedirs(output_path, exist_ok=True)

In [ ]:
et_time_series_dir = os.path.join(output_path, "et_timeseries")
config_dir = os.path.join(output_path, "config_dir")
clean(et_time_series_dir, config_dir)

### Generating start and end dates for each sliding window shift within the given period (a 7-day window with a 1-day shift)

In [ ]:
for i in window_generator(min_date_dt, max_date_dt, 7, 1):
    print(i)

In [ ]:
windows = window_generator(min_date_dt, max_date_dt, 7, 1)

### Selecting dates corresponding to the first window, that's to say the first date only

In [ ]:
window_start, window_end = next(windows)

In [ ]:
print(window_start, window_end)

### Creating the configuration dictionary with the dates and file paths associated

In [ ]:
config = create_config(
    window_start=window_start,
    window_end=window_end,
    et_single_date_dir=et_single_date_dir,
    radiation_dir=radiation_dir,
    et_time_series_dir=et_time_series_dir,
    verbose=True,
    config_dir=config_dir,
)

In [ ]:
pprint(config.model_dump())

### Running timeseries and creating the resulting et_time_series .tif file

In [ ]:
# Clean
clean(et_time_series_dir, config_dir)
logging.info("Clean output: OK")
# Read timeseries input
et_ts, radiation_ts, et_sd, dem = read_ts_input_data(config.input.model_dump())
logging.info("Read timeseries input data: OK")
# Run timeseries
updated_ts = run_timeseries(
    et_ts=et_ts,
    radiation_ts=radiation_ts,
    et_sd=et_sd,
    dem=dem,
    params=config.params.model_dump(),
    debug=config.debug.model_dump(),
)
logging.info("Update time series: OK")

### Running the global function that performs the various operations previously described, for the entire given period

In [ ]:
# Clean output
clean(et_time_series_dir, config_dir)
# Run timeseries over a period
run_window_time_series(
    period_start=min_date_dt,
    period_end=max_date_dt,
    et_single_date_dir=et_single_date_dir,
    radiation_dir=radiation_dir,
    et_time_series_dir=et_time_series_dir,
    window=7,
    shift=1,
    verbose=True,
    config_dir=config_dir,
)